SparkSession is the main entry object for PySpark. 
You can't do ANYTHING without it.

- Created ONCE per application (Singleton pattern).
- In Databricks it's ready as `spark` - no need to create it.
- Locally you must build it yourself using the builder pattern.
- Through SparkSession you read data, create DataFrames, run SQL.

KEY METHODS:

spark.read - reading data

spark.sql() - SQL queries

spark.createDataFrame() - creating DF from Python

spark.stop() - close session (release resources)

In [0]:
## ask Chris

sc = spark.sparkContext

print(sc.master)
print(sc.defaultParallelism)

## show leeson 01 - partitions

In [0]:
polish_cities = [("Warsaw", 1_800_000), ("Krakow", 800_000), ("Gdansk", 470_000)]
df_polish_cities = spark.createDataFrame(polish_cities, ["city", "population"])
df_polish_cities.show()

## transformations vs actions

Spark: The Definitive Guide

![image_1779123503157.png](./image_1779123503157.png "image_1779123503157.png")

## createDataFrame

In [0]:

## list of tuples + column names

data_tuples = [
("John", "Smith", 35, 8500.0),
("Anna", "Brown", 28, 9200.0),
("Peter", "Wilson", 42, 12000.0),
]

columns = ["first_name", "last_name", "age", "salary"]
df1 = spark.createDataFrame(data_tuples, columns)
df1.show()
df1.printSchema()

In [0]:
## list of Row objects
from pyspark.sql import Row

data_rows = [
Row(product="Laptop", price=4500, category="Electronics"),
Row(product="Desk", price=1200, category="Furniture"),
Row(product="Headphones", price=350, category="Electronics"),
]

df2 = spark.createDataFrame(data_rows)
df2.show()

In [0]:
## list of dictionaries

data_dicts = [
{"city": "Berlin", "country": "Germany", "pop": 3_700_000},
{"city": "Paris", "country": "France", "pop": 2_200_000},
{"city": "Warsaw", "country": "Poland", "pop": 1_800_000},
]
df3 = spark.createDataFrame(data_dicts)
df3.show()

In [0]:

## with explicit schema

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
StructField("first_name", StringType(), nullable=False),
StructField("age", IntegerType(), nullable=True),
StructField("salary", DoubleType(), nullable=True),
])
data = [("John", 35, 8500.0), ("Anna", 28, 9200.0)]
df4 = spark.createDataFrame(data, schema)
df4.printSchema()

In [0]:
## EMPTY df but with schema

empty_df = spark.createDataFrame([("jason",5,9.50)], schema)
empty_df.show()
print(f"Empty? {empty_df.count() == 0}") #

In [0]:
## basic inspections

print(f"Columns: {df1.columns}")

print(f"Types: {df1.dtypes}")

print(f"Row count: {df1.count()}")

print(f"Column count: {len(df1.columns)}")

first_two = df1.head(2)

print(f"First 2 from df1: {first_two}")

df1.show(truncate=False) #truncate will show full value, not first 20 chars

## explicit schema - why?

In [0]:
"""
StringType() - text
IntegerType() - int32 (-2B to 2B)
LongType() - int64 (default for ints!)
DoubleType() - float64
FloatType() - float32
BooleanType() - true/false
DateType() - date (no time)
TimestampType() - date + time
DecimalType(p,s) - precise (finance!)
ArrayType() - array
MapType() - dictionary
StructType() - nested structure
"""

from pyspark.sql import SparkSession
from pyspark.sql.types import (
StructType, StructField, StringType, IntegerType, LongType,
DoubleType, BooleanType, DateType, TimestampType, DecimalType,
ArrayType, MapType)

## example with explicit schema

In [0]:
employee_schema = StructType([
StructField("employee_id", IntegerType(), nullable=False),
StructField("name", StringType(), nullable=False),
StructField("department", StringType(), nullable=True),
StructField("salary", DoubleType(), nullable=True),
StructField("is_active", BooleanType(), nullable=False),
])

data = [
(1, "John Smith", "IT", 12000.0, True),
(2, "Anna Brown", "HR", 9500.0, True),
(3, "Peter W.", None, 8000.0, False),
]

df = spark.createDataFrame(data, employee_schema)
df.printSchema()
df.show()

## Schema with DDL - SQL like code

In [0]:
ddl_schema = "order_id INT, product STRING, quantity INT, price float"

orders = spark.createDataFrame([(1, "Laptop", 2, 4500.00), (2, "Mouse", 10, 49.99)],ddl_schema)

orders.printSchema()
orders.show()

In [0]:
## Chris - what if we have small nested and its always the same should we go with structType or we can use Variant? 

address_schema = StructType([
StructField("user_id", IntegerType()),
StructField("name", StringType()),
StructField("address", StructType([
StructField("street", StringType()),
StructField("city", StringType()),
StructField("zip", StringType()),
])),
])

nested_data = [
(1, "John", ("Main St 1", "New York", "10001")),
(2, "Anna", ("Oak Ave 5", "Chicago", "60601")),
]

df_nested = spark.createDataFrame(nested_data, address_schema)
df_nested.printSchema()
df_nested.show(truncate=False)

# Access nested fields:
df_nested.select("name", "address.city", "address.zip").show()

In [0]:
## arraytype = []
## maptype = {}


complex_schema = StructType([
StructField("id", IntegerType()),
StructField("tags", ArrayType(StringType())),
StructField("metadata", MapType(StringType(), StringType())),
])

complex_data = [
(1, ["python", "spark"], {"level": "senior", "team": "data"}),
(2, ["sql", "dbt"], {"level": "mid", "team": "analytics"}),
]

df_complex = spark.createDataFrame(complex_data, complex_schema)
df_complex.printSchema()
df_complex.show(truncate=False)

In [0]:
## Chris - can we use Schema as DDL to actually use it  ?

xtracted_schema = df.schema
print(f"Schema as object: {extracted_schema}")
print(f"Schema as DDL: {extracted_schema.simpleString()}")

## Chris: 

Is it True? 

EXAMPLE 6: 

inferSchema vs explicit (performance comparison)

inferSchema=True - Spark must read data TWICE:
1. once to guess types
2. once to load data

On large files this DOUBLES read time!

In production:

df = spark.read.schema(my_schema).json("path/to/data/") <- NO inferSchema

## instead of throwing errors

Replaces with Null: Fields that cannot be parsed are replaced with null.

Preserves Rows: The row itself is kept, allowing data processing to continue uninterrupted.

.option("mode", "PERMISSIVE")

------
skips bad rows

.option("mode", "DROPMALFORMED")

-----
throws exception

.option("mode","FAILFAST")

## Chris - why it's idempotent ? 

IDEMPOTENT WRITE (PRODUCTION):

.mode("overwrite").option("replaceWhere", "date = '2025-01-01'")

- overwrites ONLY the given partition, not the whole folder

## Chris - if I have a lot of files per day should I partition per day ? does it depend on size or amount ? 

PARTITIONING:

.partitionBy("year", "month")

- physical split into folders: /year=2024/month=01/file.parquet
- huge speedup for filtering (partition pruning)

In [0]:
## Chris - do I need to know how many exceutors I have for repartition?

# Scala wszystko do 1 partycji → zapisze 1 plik parquet
df.coalesce(1).write.mode("overwrite").parquet(f"{DATA_DIR}/single_file")

# Rozdziela dane na 4 partycje → zapisze 4 pliki parquet
df.repartition(4).write.mode("overwrite").parquet(f"{DATA_DIR}/four_files")


# Kiedy co:

# coalesce(1) — chcesz jeden plik (np. mały export, CSV do pobrania). Uwaga: przy dużych danych wrzucasz wszystko na jeden executor → OOM risk.

# repartition(n) — chcesz kontrolowany, równomierny podział (np. przed zapisem do Delta/Parquet w produkcji).

## Chris
WHEN TO USE WHICH?
- String -> quick, readable, but doesn't work with expressions
- col() -> RECOMMENDED in production, works with transformations -> there will  be no ambiguiity with col() ?
- df.col -> problematic with joins (ambiguous!)

In [0]:
from pyspark.sql.functions import lit, col

data = [
(1, "John", "Smith", "IT", 12000.0),
(2, "Anna", "Brown", "HR", 9500.0),
(3, "Peter", "Wilson", "IT", 11000.0),
(4, "Maria", "Green", "Sales", 8500.0),
]
df = spark.createDataFrame(data, ["id", "first_name", "last_name", "dept", "salary"])

df.select(
col("first_name"),
lit("USD").alias("currency"),
lit(2025).alias("year"), # column with constant value!!
col("salary"),
).show()